In [1]:
import langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

Example 1 : Simple LLM call With streaming

In [4]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

In [29]:
model = init_chat_model("groq:llama-3.1-8b-instant")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11f95be00>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x12901d2e0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr(''), groq_api_base=None, groq_proxy=None)

In [5]:
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI

llm = ChatGroq(model="llama-3.1-8b-instant")
llm

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11824b950>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1188d04d0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [6]:
# Create messages
messages = [
    SystemMessage("You are a helpful AI assistant"),
    HumanMessage("What are the top 2 benefits of using Langchain?")
]

In [9]:
#invoke the model
response = llm.invoke(messages)
# response = model.invoke(messages)
response

AIMessage(content="Langchain is an AI platform that enables users to build custom large language models (LLMs) and integrate them with other tools and services. Based on my knowledge, the top 2 benefits of using Langchain are:\n\n1. **Customizable and Domain-Specific Models**: Langchain allows users to create custom LLMs tailored to their specific use case, industry, or domain. This means you can fine-tune a model to perform better on tasks relevant to your business or project, leading to more accurate and relevant results. By doing so, you can leverage the power of AI to solve complex problems that may not be addressed by general-purpose LLMs.\n\n2. **Modular and Integratable**: Langchain's architecture is designed to be modular and integratable with other tools and services. This enables users to easily incorporate LLMs into their workflows, whether it's a web application, a chatbot, or a data analysis pipeline. By integrating LLMs with other technologies, users can unlock new insigh

In [10]:
print(response.content)

Langchain is an AI platform that enables users to build custom large language models (LLMs) and integrate them with other tools and services. Based on my knowledge, the top 2 benefits of using Langchain are:

1. **Customizable and Domain-Specific Models**: Langchain allows users to create custom LLMs tailored to their specific use case, industry, or domain. This means you can fine-tune a model to perform better on tasks relevant to your business or project, leading to more accurate and relevant results. By doing so, you can leverage the power of AI to solve complex problems that may not be addressed by general-purpose LLMs.

2. **Modular and Integratable**: Langchain's architecture is designed to be modular and integratable with other tools and services. This enables users to easily incorporate LLMs into their workflows, whether it's a web application, a chatbot, or a data analysis pipeline. By integrating LLMs with other technologies, users can unlock new insights, automate tasks, and

In [11]:
## streaming example
for chunk in llm.stream(messages):
    print(chunk.content, end="",flush=True)

Langchain is an open-source platform for building conversational AI models. Based on my knowledge, the top 2 benefits of using Langchain are:

1. **Improved Conversational Flow**: Langchain allows developers to build more conversational and context-aware AI models by integrating multiple AI models and tools into a single platform. This enables more natural and human-like conversations, which can be beneficial for applications such as customer service chatbots, virtual assistants, and language translation tools.

2. **Enhanced Model Flexibility**: Langchain provides a flexible and modular architecture that enables developers to easily integrate and combine different AI models, such as language models, knowledge graphs, and databases. This flexibility allows developers to build custom AI models that can be tailored to specific use cases and domains, which can be beneficial for applications such as content creation, research assistance, and decision support systems.

Please note that thes

# Dynamic Prompt templates

In [17]:
from langchain_core.prompts import ChatPromptTemplate

## create translation app

translation_template = ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. Translate the following {text} to {target_language} maintaining the tone and style."),
    ("human", "{text}"),
])

## using the template
prompt = translation_template.invoke({
    "source_language": "English",
    "target_language": "Spanish",
    "text": "Langchain makes building AI application incredibly easy!"
})

In [19]:
llm.invoke(prompt)

AIMessage(content='Langchain hace que crear aplicaciones de inteligencia artificial sea extremadamente fácil.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 70, 'total_tokens': 87, 'completion_time': 0.069930117, 'completion_tokens_details': None, 'prompt_time': 0.017961286, 'prompt_tokens_details': None, 'queue_time': 0.052169617, 'total_time': 0.087891403}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dc42d-9a18-79e1-a428-2928b6bbf84a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 70, 'output_tokens': 17, 'total_tokens': 87})

# Building your First Chain

In [ ]:
from langchain_core.output_parsers import StrOutputParser

def create_story_chain():
    story_prompt = ChatPromptTemplate.from_messages(
        [
        ("system", "You are a creative storyteller. Write a short story based on the following"),
         ("user", "Theme: {theme}\n Main characters: {characters}\nSetting: {setting}"),
        ]
    )
    ## Template for the story Analysis
    analysis_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a literary critic. Analyze the following story and provide insights."),
    ("user", "{story}")
])
    
    story_chain=(
        story_prompt| model | StrOutputParser
    )
    analysis_chain=(
        {"story": story_chain} | analysis_prompt | model | StrOutputParser()
    )
    return analysis_chain

In [14]:
chain=create_story_chain()
chain

{
  story: ChatPromptTemplate(input_variables=['characters', 'setting', 'theme'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a creative storyteller. Write a short story based on the following.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['characters', 'setting', 'theme'], input_types={}, partial_variables={}, template='Theme: {theme}\nMain characters: {characters}\nSetting: {setting}'), additional_kwargs={})])
         | ChatOpenAI(output_version=None, profile={'name': 'GPT-3.5-turbo', 'release_date': '2023-03-01', 'last_updated': '2023-11-06', 'open_weights': False, 'max_input_tokens': 16385, 'max_output_tokens': 4096, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning